In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
from torchvision import transforms
from tqdm import tqdm

from gap.classifiers import ScalableResNetLite
from gap.data import OxfordPetDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
transform_train = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),

    transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
    ),
])

transform_test = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),

    transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
    ),
])

root = "../datasets/oxford-iiit-pet/images/images"
#root = "../oxford-iiit-pet/images"

full_dataset = OxfordPetDataset(root, binary=True, transform=transform_train)
num_classes = len(full_dataset.classes)
print("Classes:", num_classes)

In [ ]:
total = len(full_dataset)
train_size = int(0.7 * total)
val_size = int(0.15 * total)
test_size = total - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(42) 
)

# Validation/test use deterministic transforms
val_dataset.dataset.transform = transform_test
test_dataset.dataset.transform = transform_test

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

print("Train:", len(train_dataset), "Val:", len(val_dataset), "Test:", len(test_dataset))

In [ ]:
def make_optimizer(opt_name, model, lr, wd):
    if opt_name == "sgd":
        return optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    else:
        return optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    correct, total, running_loss = 0, 0, 0.0

    # Wrap loader in tqdm for a live progress bar
    for x, y in tqdm(loader, desc="Training", leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return correct / total, running_loss / total


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Evaluating", leave=False):
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

In [ ]:
# -----------------------------------------
# Use a smaller subset for sensitivity analysis
# -----------------------------------------
sa_subset_size = 1000  # you can lower to 500 if needed

sa_train_subset, _ = random_split(
    train_dataset,
    [sa_subset_size, len(train_dataset) - sa_subset_size], generator=torch.Generator().manual_seed(42) 
)

train_loader_sa = DataLoader(
    sa_train_subset, batch_size=batch_size, shuffle=True,
    num_workers=0, pin_memory=True
)

In [ ]:
#next(iter(train_loader_sa))

In [ ]:
len(full_dataset.files)

In [ ]:
lrs = [1e-4, 3e-4, 1e-3]
channels_list = [48, 64, 96]
depth_list = [4, 5, 6]
opts = ["sgd", "adam"]
weight_decays = [0, 1e-4, 5e-4]

criterion = nn.CrossEntropyLoss()

coarse_results = []

for lr in lrs:
    for ch in channels_list:
        for d in depth_list:
            for opt in opts:
                for wd in weight_decays:
                    print("\nStarting config:", lr, ch, d, opt, wd)
                    model = ScalableResNetLite(channels=ch, depth=d, num_classes=num_classes).to(device)
                    optimizer = make_optimizer(opt, model, lr, wd)

                    print("Traiining for 1 epoch...")
                    train_acc, _ = train_one_epoch(model, train_loader_sa, criterion, optimizer)
                    print("Evaluating...")
                    val_acc = evaluate(model, val_loader)

                    coarse_results.append(
                        ({"lr": lr, "channels": ch, "depth": d, "opt": opt, "wd": wd},
                         val_acc)
                    )
                    print("SA:", lr, ch, d, opt, wd, "val_acc=", val_acc)

coarse_results.sort(key=lambda x: x[1], reverse=True)

In [ ]:
def derive_refined_space(coarse_results, top_frac=0.2):
    top_k = max(1, int(len(coarse_results) * top_frac))
    top_configs = [cfg for (cfg, acc) in coarse_results[:top_k]]

    lrs = [c["lr"] for c in top_configs]
    channels = [c["channels"] for c in top_configs]
    depths = [c["depth"] for c in top_configs]
    opts = [c["opt"] for c in top_configs]
    wds = [c["wd"] for c in top_configs]

    return {
        "lr": (min(lrs), max(lrs)),
        "channels": sorted(set(channels)),
        "depth": sorted(set(depths)),
        "opt": sorted(set(opts)),
        "wd": sorted(set(wds)),
    }

refined_space = derive_refined_space(coarse_results)
print("Refined space:", refined_space)

In [ ]:
def sample_config(space):
    lr_min, lr_max = space["lr"]
    lr = 10 ** random.uniform(np.log10(lr_min), np.log10(lr_max))
    ch = random.choice(space["channels"])
    d = random.choice(space["depth"])
    opt = random.choice(space["opt"])
    wd = random.choice(space["wd"])
    return {"lr": lr, "channels": ch, "depth": d, "opt": opt, "wd": wd}

In [ ]:
num_trials = 20
rand_results = []

for t in range(num_trials):
    config = sample_config(refined_space)
    model = ScalableResNetLite(channels=config["channels"], depth=config["depth"], num_classes=num_classes).to(device)
    optimizer = make_optimizer(config["opt"], model, config["lr"], config["wd"])

    train_acc, _ = train_one_epoch(model, train_loader, criterion, optimizer)
    val_acc = evaluate(model, val_loader)

    rand_results.append((config, val_acc))
    print("RS trial", t, config, "val_acc=", val_acc)

rand_results.sort(key=lambda x: x[1], reverse=True)
best_config, best_val = rand_results[0]
print("Best config:", best_config, "val_acc=", best_val)

In [ ]:
final_model = ScalableResNetLite(channels=best_config["channels"], depth=best_config["depth"], num_classes=num_classes).to(device)
optimizer = make_optimizer(best_config["opt"], final_model,
                           best_config["lr"], best_config["wd"])
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

best_val = 0.0
num_epochs = 50
for epoch in range(num_epochs):
    train_acc, train_loss = train_one_epoch(final_model, train_loader, criterion, optimizer)
    val_acc = evaluate(final_model, val_loader)
    if val_acc > best_val:
        best_val = val_acc
        torch.save(final_model.state_dict(), "../saved_models/pets_binary_classifier.pth")
    print(epoch, "train_acc=", train_acc, "val_acc=", val_acc)

    scheduler.step()

final_model.load_state_dict(torch.load("../saved_models/pets_binary_classifier.pth"))
test_acc = evaluate(final_model, test_loader)
print("Final test accuracy:", test_acc)

In [ ]:
num_epochs = 18
for epoch in range(num_epochs):
    train_acc, train_loss = train_one_epoch(final_model, train_loader, criterion, optimizer)
    val_acc = evaluate(final_model, val_loader)
    print(epoch, "train_acc=", train_acc, "val_acc=", val_acc)

test_acc = evaluate(final_model, test_loader)
print("Final test accuracy:", test_acc)

In [ ]:
import json
import numpy as np
#best_config = {'lr': np.float64(0.000108995648041844), 'channels': 64, 'depth': 6, 'opt': 'sgd', 'wd': 0.0001}
metadata = {
    "model_config": {
        "lr": float(best_config["lr"]),
        "channels": int(best_config["channels"]),
        "depth": int(best_config["depth"]),
        "optimizer": best_config["opt"],
        "weight_decay": float(best_config["wd"]),
    },
    "test_accuracy": float(test_acc),
}


with open("../saved_models/pets_binary_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)